# Getting Started with Taipy Scenarios & Data Management

This tutorial demonstrates how to transform existing data processing workflows into Taipy scenarios. 

## Traditional User Data and Functions

We'll start with traditional Python code for some data workflow.

### Loading Sample Data

First, let's load our sample dataset which contains time-series data with dates and values.

In [ ]:
import datetime as dt
from pathlib import Path

import pandas as pd

In [ ]:
dataset_df = pd.read_csv("../dataset.csv")
dataset_df.head()

### User Functions

These functions represent typical data processing steps in a machine learning pipeline:
1. **Data Cleaning**: Standardize and filter the dataset
2. **Prediction**: Generate forecasts based on historical data  
3. **Evaluation**: Measure prediction accuracy using mean squared error

In [ ]:
def clean_data(initial_dataset: pd.DataFrame):
    print("     Cleaning data")
    initial_dataset["Date"] = pd.to_datetime(initial_dataset["Date"])
    cleaned_dataset = initial_dataset[["Date", "Value"]]
    return cleaned_dataset


def predict(cleaned_dataset: pd.DataFrame, day: dt.datetime):
    print("     Predicting")
    train_dataset = cleaned_dataset[cleaned_dataset["Date"] < pd.Timestamp(day)]
    predictions = train_dataset["Value"][-30:].reset_index(drop=True)
    date_range = pd.date_range(start=pd.Timestamp(day), periods=30, freq="D")
    smooth_predictions = predictions.rolling(window=3).mean()
    smooth_predictions = smooth_predictions.round(2)
    return pd.DataFrame({"Date": date_range, "Prediction": predictions, "Smooth Prediction": smooth_predictions})


def evaluate(predictions, cleaned_dataset, day):
    print("     Evaluating")
    expected = cleaned_dataset.loc[cleaned_dataset["Date"] >= pd.Timestamp(day), "Value"][:30].reset_index(drop=True)
    mse = ((predictions["Prediction"] - expected) ** 2).mean()
    return int(mse)

### Traditional Pipeline Execution

Here's how you would typically run this workflow without Taipy - calling each function in sequence.

In [ ]:
today = dt.datetime(2021, 7, 26)
cleaned_dataset = clean_data(dataset_df)
predictions = predict(cleaned_dataset, today)
evaluation = evaluate(predictions, cleaned_dataset, today)
print(f"Mean Squared Error: {evaluation}")

## Transforming to Taipy: Scenario & Data Management

Now let's see how Taipy can enhance this workflow by providing:
- **Automated data persistence** (no more manual file saves)
- **Smart task execution** (skip unchanged computations)
- **Workflow orchestration** (manage complex pipelines)

### Step 1: Configuration Setup

In Taipy, scenarios represent some business process or workflow. Scenarios are defined through configuration objects that act as blueprints:
- **Data Node Configs**: Define where and how data is stored and loaded
- **Task Configs**: Define what functions to execute and their inputs/outputs  
- **Scenario Configs**: Orchestrate tasks into complete workflows

In [ ]:
from taipy import Config, Frequency, Scope

In [ ]:
# Data node configs
initial_dataset_cfg = Config.configure_data_node(
    id="initial_dataset",
    storage_type="csv",
    path="../dataset.csv",
    scope=Scope.GLOBAL,
)
day_cfg = Config.configure_data_node(id="day", default_data=dt.datetime(2021, 7, 26))
cleaned_dataset_cfg = Config.configure_data_node(id="cleaned_dataset", storage_type="parquet", scope=Scope.GLOBAL)
predictions_cfg = Config.configure_data_node(id="predictions")
evaluation_cfg = Config.configure_data_node(id="evaluation")

In [ ]:
# Task configs
clean_data_task_cfg = Config.configure_task(
    id="clean_data",
    function=clean_data,
    input=initial_dataset_cfg,
    output=cleaned_dataset_cfg,
    skippable=True,
)

predict_task_cfg = Config.configure_task(
    id="predict",
    function=predict,
    input=[cleaned_dataset_cfg, day_cfg],
    output=predictions_cfg,
    skippable=True,
)

evaluate_task_cfg = Config.configure_task(
    id="evaluate",
    function=evaluate,
    input=[predictions_cfg, cleaned_dataset_cfg, day_cfg],
    output=evaluation_cfg,
    skippable=True,
)

In [ ]:
# Scenario config
scenario_cfg = Config.configure_scenario(
    id="scenario",
    task_configs=[clean_data_task_cfg, predict_task_cfg, evaluate_task_cfg],
    frequency=Frequency.MONTHLY,
)

In [ ]:
# (Optional) Export the config to a toml file for visualization
Config.export("config.toml")

### Step 2: Running Taipy Scenarios

Now that we have configured our scenario, let's see how to create and execute scenario instances. 

In [ ]:
import taipy as tp

In [ ]:
# Imports for annotations
from taipy import DataNode, Scenario

In [ ]:
# Start the Taipy Orchestrator service to manage task execution
orchestrator = tp.Orchestrator()
orchestrator.run()

#### Creating and Submitting Scenarios

Create a scenario instance from our configuration and submit it for execution.

In [ ]:
# Create a scenario instance from our configuration blueprint
scenario_1 = tp.create_scenario(scenario_cfg)
assert isinstance(scenario_1, Scenario)
# Execute the scenario, which will run its tasks as needed
submission = scenario_1.submit()

#### Reading and Writing to Data Nodes

In [ ]:
# Access data nodes through the scenario instance
assert isinstance(scenario_1.data_nodes["day"], DataNode)
# Method 1: Access via the data_nodes dictionary
print(scenario_1.data_nodes["day"])
# Method 2: Access directly as a scenario attribute
print(scenario_1.day)

In [ ]:
# You can read from and write to data nodes using the `read` and `write` methods respectively
print("Old day:", scenario_1.data_nodes["day"].read())
scenario_1.data_nodes["day"].write(dt.datetime(2021, 9, 30))  # Change prediction date
print("New day:", scenario_1.data_nodes["day"].read())

In [ ]:
# Demonstrate intelligent task skipping - only affected tasks will re-run
resubmission = scenario_1.submit()
# Since we updated the "day" data node, only the "predict" and "evaluate" tasks were rerun
# The "clean_data" task was skipped because its input (initial_dataset) hasn't changed

#### Understanding Data Node Scopes

Data node **scopes** control how data is shared across scenarios. There are three scope levels:
- **GLOBAL**: Shared across all scenarios (useful for reference data like datasets)
- **CYCLE**: Shared within scenarios of the same frequency cycle (e.g., monthly scenarios)  
- **SCENARIO**: Isolated to individual scenarios (default behavior)

Understanding scopes helps optimize data storage and ensures proper data sharing patterns.

In [ ]:
# "initial_dataset_cfg" and "cleaned_dataset_cfg" have a GLOBAL scope, so they are shared across all scenarios
initial_dataset_cfg.scope

In [ ]:
# If we create another scenario, it will reuse the same data nodes for "initial_dataset" and "cleaned_dataset"
tp.create_scenario(scenario_cfg, name="Another Scenario")
# Notice in the list below that initial_dataset and cleaned_dataset have only one instance each, shared by all scenarios
tp.get_data_nodes()

#### Understanding Cycles

**Cycles** represent time periods that group related scenarios together. They're determined by:
- The scenario's **frequency** (daily, weekly, monthly, quarterly, yearly)
- The **creation date** of the scenario

Cycles enable:
- **Temporal organization**: Group scenarios by business periods (e.g., monthly reports)
- **Data isolation**: CYCLE-scoped data nodes are shared only within the same time period

In [ ]:
# Check which cycle (time period) our first scenario belongs to
# Since we configured MONTHLY frequency, this shows the current month
scenario_1.cycle.get_label()

In [ ]:
# Create a scenario for a different time period by manually specifying creation_date at initialization
scenario_january_2025 = tp.create_scenario(scenario_cfg, creation_date=dt.datetime(2025, 1, 7))

# This scenario belongs to a different monthly cycle than our first scenario
scenario_january_2025.cycle.get_label()